## Meta Combiner 
(_tracks_model_a + lyrics_model_b_)

Stack **LogisticRegression** on `[p_a, p_b (filled), has_lyrics]`, compare a fixed **gated blend** on the test split, write **`preds_ensemble.parquet`**.

**Inputs:** 
- `preds_model_a.parquet` (from `01e_tracks_model_a.ipynb`)
- `preds_model_b.parquet` (from `02c_lyrics_model_b.ipynb`)

**Output:** 
- `data/processed/preds_ensemble.parquet` — not `preds_model_a` (that is produced only in `01e`)


In [10]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

PREDS_A = Path("../data/processed/preds_model_a.parquet")
PREDS_B = Path("../data/processed/preds_model_b.parquet")
OUT_ENS = Path("../data/processed/preds_ensemble.parquet")

preds_a = pd.read_parquet(PREDS_A)
B = pd.read_parquet(PREDS_B)
assert len(preds_a) == len(B)
assert (preds_a["row_id"].values == B["row_id"].values).all()
assert (preds_a["track_id"].values == B["track_id"].values).all()
y = preds_a["viral"].astype(int).values


In [11]:
meta = preds_a.merge(
    B[["row_id", "has_lyrics", "p_b"]], on="row_id", how="inner", validate="1:1"
)
meta["p_b_filled"] = meta["p_b"].fillna(0.5)
meta_X = meta[["p_a", "p_b_filled", "has_lyrics"]]
meta_train = (meta["split"] == "val").values & meta["has_lyrics"].values
meta_test = (meta["split"] == "test").values

comb = LogisticRegression(max_iter=2000)
comb.fit(meta_X.loc[meta_train], y[meta_train])

p_stack = comb.predict_proba(meta_X)[:, 1]
w = 0.7
p_gate = np.where(
    meta["has_lyrics"].values,
    w * meta["p_a"].values + (1 - w) * meta["p_b_filled"].values,
    meta["p_a"].values,
)
print(
    "STACK test  ROC-AUC", roc_auc_score(y[meta_test], p_stack[meta_test]),
    "PR-AUC", average_precision_score(y[meta_test], p_stack[meta_test]),
)
print(
    "GATE  test  ROC-AUC", roc_auc_score(y[meta_test], p_gate[meta_test]),
    "PR-AUC", average_precision_score(y[meta_test], p_gate[meta_test]),
)

full = preds_a.merge(B[["row_id", "has_lyrics", "p_b"]], on="row_id", how="left")
full["has_lyrics"] = full["has_lyrics"].fillna(False)
full["p_b_filled"] = full["p_b"].fillna(0.5)
full["p_ens"] = comb.predict_proba(full[["p_a", "p_b_filled", "has_lyrics"]])[:, 1]

OUT_ENS.parent.mkdir(parents=True, exist_ok=True)
cols = ["row_id", "track_id", "split", "viral", "has_lyrics", "p_a", "p_b", "p_ens"]
full[cols].to_parquet(OUT_ENS, index=False)
print("Wrote", OUT_ENS)


STACK test  ROC-AUC 0.8499036415949902 PR-AUC 0.6753159975113845
GATE  test  ROC-AUC 0.8546960271271966 PR-AUC 0.6796580297294895
Wrote ../data/processed/preds_ensemble.parquet
